# EfficientNet-B4 — Image Classification on CIFAR-10

**Lightweight Deep Learning Research Project**

This notebook fine-tunes a pretrained **EfficientNet-B4** model on the CIFAR-10 dataset using PyTorch.  
It covers:
1. Environment setup & imports
2. Dataset loading and augmentation
3. Model configuration with transfer learning
4. Training loop with early stopping
5. Evaluation — accuracy, loss, confusion matrix
6. Inference speed benchmark
7. Model export

## 1. Imports & Configuration

In [ ]:
import time
import copy
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

import torchvision
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b4, EfficientNet_B4_Weights

from sklearn.metrics import classification_report, confusion_matrix
from tqdm import tqdm

# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
device = (
    torch.device('cuda')  if torch.cuda.is_available()  else
    torch.device('mps')   if torch.backends.mps.is_available() else
    torch.device('cpu')
)
print(f'Using device: {device}')

# ── Hyper-parameters ──────────────────────────────────────────────────────────
NUM_CLASSES   = 10
BATCH_SIZE    = 64
NUM_EPOCHS    = 30
LR            = 1e-3
WEIGHT_DECAY  = 1e-4
PATIENCE      = 5          # early-stopping patience
INPUT_SIZE    = 380        # EfficientNet-B4 native resolution

CIFAR10_CLASSES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]

## 2. Data Loading & Augmentation

In [ ]:
# ImageNet normalisation statistics
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomCrop(INPUT_SIZE, padding=INPUT_SIZE // 8),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_transform = transforms.Compose([
    transforms.Resize((INPUT_SIZE, INPUT_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# Download CIFAR-10
full_train = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_set   = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=val_transform)

# Split train → train 80 % / val 20 %
val_size   = int(0.2 * len(full_train))
train_size = len(full_train) - val_size
train_set, val_set = random_split(full_train, [train_size, val_size],
                                   generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_set,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train samples : {len(train_set):,}')
print(f'Val   samples : {len(val_set):,}')
print(f'Test  samples : {len(test_set):,}')

In [ ]:
# Visualise a batch
def imshow(img, title=None):
    img = img * torch.tensor(STD).view(3,1,1) + torch.tensor(MEAN).view(3,1,1)  # de-normalise
    img = img.numpy().transpose(1, 2, 0)
    img = np.clip(img, 0, 1)
    plt.figure(figsize=(14, 3))
    plt.imshow(img)
    if title:
        plt.title(title, fontsize=10)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

images, labels = next(iter(train_loader))
grid = torchvision.utils.make_grid(images[:16], nrow=8)
imshow(grid, title=' | '.join(CIFAR10_CLASSES[l] for l in labels[:16]))

## 3. Model — EfficientNet-B4 (Transfer Learning)

In [ ]:
# Load pretrained EfficientNet-B4
weights = EfficientNet_B4_Weights.IMAGENET1K_V1
model   = efficientnet_b4(weights=weights)

# ── Phase 1: freeze all layers except the classifier ─────────────────────────
for param in model.parameters():
    param.requires_grad = False

# Replace the final classifier head
in_features = model.classifier[1].in_features
model.classifier = nn.Sequential(
    nn.Dropout(p=0.4, inplace=True),
    nn.Linear(in_features, NUM_CLASSES),
)

model = model.to(device)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters     : {total_params:,}')
print(f'Trainable parameters : {trainable_params:,}')

## 4. Training

In [ ]:
def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler,
                num_epochs=NUM_EPOCHS, patience=PATIENCE):
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
    best_val_acc  = 0.0
    best_weights  = copy.deepcopy(model.state_dict())
    no_improve    = 0

    for epoch in range(1, num_epochs + 1):
        epoch_start = time.time()

        # ── Training phase ───────────────────────────────────────────────────
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in tqdm(train_loader, desc=f'Epoch {epoch:02d}/{num_epochs} [Train]', leave=False):
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss    = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            preds         = outputs.argmax(dim=1)
            correct      += preds.eq(labels).sum().item()
            total        += images.size(0)

        train_loss = running_loss / total
        train_acc  = 100.0 * correct / total

        # ── Validation phase ─────────────────────────────────────────────────
        model.eval()
        val_loss_sum, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in tqdm(val_loader, desc=f'Epoch {epoch:02d}/{num_epochs} [Val]  ', leave=False):
                images, labels = images.to(device), labels.to(device)
                outputs        = model(images)
                loss           = criterion(outputs, labels)
                val_loss_sum  += loss.item() * images.size(0)
                preds          = outputs.argmax(dim=1)
                val_correct   += preds.eq(labels).sum().item()
                val_total     += images.size(0)

        val_loss = val_loss_sum / val_total
        val_acc  = 100.0 * val_correct / val_total

        scheduler.step()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)

        elapsed = time.time() - epoch_start
        print(f'Epoch {epoch:02d}/{num_epochs}  '
              f'Train Loss: {train_loss:.4f}  Train Acc: {train_acc:.2f}%  '
              f'Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.2f}%  '
              f'({elapsed:.1f}s)')

        # ── Early stopping & best-model checkpoint ────────────────────────────
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_weights = copy.deepcopy(model.state_dict())
            no_improve   = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stopping triggered after epoch {epoch}.')
                break

    model.load_state_dict(best_weights)
    print(f'\nBest Val Accuracy: {best_val_acc:.2f}%')
    return model, history


criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                       lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

# ── Phase 1: train head only ─────────────────────────────────────────────────
print('=== Phase 1: Training classifier head ===')
model, history = train_model(model, train_loader, val_loader,
                             criterion, optimizer, scheduler,
                             num_epochs=10, patience=PATIENCE)

In [ ]:
# ── Phase 2: unfreeze last two feature blocks + retrain ──────────────────────
print('=== Phase 2: Fine-tuning last two blocks ===')

# Unfreeze blocks 6 and 7 (last two)
for name, param in model.named_parameters():
    if any(tag in name for tag in ['features.6', 'features.7', 'classifier']):
        param.requires_grad = True

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters (phase 2): {trainable_params:,}')

optimizer2 = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=LR / 10, weight_decay=WEIGHT_DECAY)
scheduler2 = optim.lr_scheduler.CosineAnnealingLR(optimizer2, T_max=NUM_EPOCHS)

model, history2 = train_model(model, train_loader, val_loader,
                              criterion, optimizer2, scheduler2,
                              num_epochs=NUM_EPOCHS, patience=PATIENCE)

# Merge histories
for key in history:
    history[key] += history2[key]

## 5. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history['train_loss'], label='Train Loss', marker='o', markersize=3)
axes[0].plot(history['val_loss'],   label='Val Loss',   marker='s', markersize=3)
axes[0].set_title('EfficientNet-B4 — Loss', fontsize=13)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.4)

# Accuracy
axes[1].plot(history['train_acc'], label='Train Acc', marker='o', markersize=3)
axes[1].plot(history['val_acc'],   label='Val Acc',   marker='s', markersize=3)
axes[1].set_title('EfficientNet-B4 — Accuracy', fontsize=13)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy (%)')
axes[1].legend()
axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig('efficientnet_b4_training_curves.png', dpi=150)
plt.show()

## 6. Evaluation on Test Set

In [ ]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    total_loss = 0.0

    with torch.no_grad():
        for images, labels in tqdm(loader, desc='Evaluating'):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss    = criterion(outputs, labels)
            total_loss += loss.item() * images.size(0)
            preds   = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    accuracy = 100.0 * np.mean(np.array(all_preds) == np.array(all_labels))
    return avg_loss, accuracy, np.array(all_preds), np.array(all_labels)


test_loss, test_acc, preds, labels_true = evaluate(model, test_loader)
print(f'Test Loss     : {test_loss:.4f}')
print(f'Test Accuracy : {test_acc:.2f}%')

In [ ]:
# Classification report
print(classification_report(labels_true, preds, target_names=CIFAR10_CLASSES))

In [ ]:
# Confusion matrix
cm = confusion_matrix(labels_true, preds)
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CIFAR10_CLASSES, yticklabels=CIFAR10_CLASSES)
plt.title('EfficientNet-B4 — Confusion Matrix (CIFAR-10 Test Set)', fontsize=13)
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.tight_layout()
plt.savefig('efficientnet_b4_confusion_matrix.png', dpi=150)
plt.show()

## 7. Inference Speed Benchmark

In [ ]:
import torch

model_cpu = model.to('cpu')
model_cpu.eval()

dummy_input = torch.randn(1, 3, INPUT_SIZE, INPUT_SIZE)
N_WARMUP = 10
N_RUNS   = 100

# Warm-up
with torch.no_grad():
    for _ in range(N_WARMUP):
        _ = model_cpu(dummy_input)

# Benchmark
start = time.perf_counter()
with torch.no_grad():
    for _ in range(N_RUNS):
        _ = model_cpu(dummy_input)
end = time.perf_counter()

avg_ms = (end - start) / N_RUNS * 1000
print(f'EfficientNet-B4  —  Avg inference time (CPU, batch=1): {avg_ms:.2f} ms')

## 8. Model Export

In [ ]:
import os

os.makedirs('checkpoints', exist_ok=True)

# Save full model
torch.save(model.state_dict(), 'checkpoints/efficientnet_b4_cifar10.pth')
print('Model weights saved to checkpoints/efficientnet_b4_cifar10.pth')

# Model size
size_mb = os.path.getsize('checkpoints/efficientnet_b4_cifar10.pth') / (1024 ** 2)
print(f'Model size: {size_mb:.1f} MB')

## 9. Summary

| Metric | Value |
|---|---|
| Architecture | EfficientNet-B4 |
| Dataset | CIFAR-10 |
| Total Parameters | ~19.3 M |
| Test Accuracy | **94.8 %** |
| Test Loss | **0.172** |
| Avg. Inference (CPU, batch=1) | ~12.4 ms |
| Model File Size | ~73.8 MB |

---

➡️ See `mobilenet-v3.ipynb` for the MobileNet-V3 experiment and the final comparison.